<a href="https://colab.research.google.com/github/Maverick-Ansh/intentions_emergent/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess, sys, os, platform
print("py", sys.version.split()[0], "|", platform.platform()[:40])
try:
    out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,compute_cap,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=60)
    print("GPU:", out.stdout.strip() or out.stderr.strip()[:200])
except Exception as e:
    print("nvidia-smi failed:", e)
try:
    import torch
    print("torch", torch.__version__, "cuda", torch.version.cuda, "n_gpu", torch.cuda.device_count())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"  dev0 {p.name} {p.total_memory/1e9:.1f}GB sm{p.major}{p.minor} bf16={torch.cuda.is_bf16_supported()}")
except Exception as e:
    print("torch:", e)
import shutil
print("disk free %.1f GB" % (shutil.disk_usage('/').free/1e9))
print("RAM:", subprocess.run(["free","-g"], capture_output=True, text=True).stdout.splitlines()[1])


py 3.12.13 | Linux-6.12.90+-x86_64-with-glibc2.35
GPU: Tesla T4, 15360 MiB, 7.5, 580.159.04
Tesla T4, 15360 MiB, 7.5, 580.159.04
torch 2.10.0+cu128 cuda 12.8 n_gpu 2
  dev0 Tesla T4 15.6GB sm75 bf16=True
disk free 1100.2 GB
RAM: Mem:              31           1          26           0           3          29


In [2]:
import importlib.metadata as md
for p in ["transformers","peft","datasets","accelerate","trl","sentence-transformers",
          "bitsandbytes","scipy","huggingface-hub","langchain-typesafe","langchain-core"]:
    try: print(f"{p:24s} {md.version(p)}")
    except Exception: print(f"{p:24s} -- MISSING")


transformers             5.0.0
peft                     0.18.1
datasets                 4.8.5
accelerate               1.13.0
trl                      -- MISSING
sentence-transformers    5.4.0
bitsandbytes             -- MISSING
scipy                    1.16.3
huggingface-hub          1.10.1
langchain-typesafe       -- MISSING
langchain-core           1.2.28


In [3]:
%pip install -q langchain-typesafe 2>&1 | tail -3
print("--- what does the JEV client actually expose? ---")
try:
    import langchain_typesafe as lts, inspect, importlib.metadata as md
    print("version:", md.version("langchain-typesafe"))
    print("exports:", [n for n in dir(lts) if not n.startswith("_")])
    for name in ["Noul","Choice","Score","TypeSafeClassifier"]:
        obj = getattr(lts, name, None)
        if obj is None: print(f"\n{name}: NOT PRESENT"); continue
        try: sig = str(inspect.signature(obj))
        except Exception as e: sig = f"<{e}>"
        print(f"\n{name}{sig}")
        doc = (inspect.getdoc(obj) or "").strip().splitlines()[:6]
        for l in doc: print("   ", l)
except Exception as e:
    print("import failed:", type(e).__name__, e)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
Note: you may need to restart the kernel to use updated packages.
--- what does the JEV client actually expose? ---
version: 0.0.1a2
exports: ['Answer', 'Choice', 'ChoiceAnswer', 'Noul', 'NoulAnswer', 'NoulCriteria', 'Question', 'Score', 'ScoreAnswer', 'State', 'TypeSafeClassifier', 'Usage', 'classifier', 'client', 'types']

Noul(*, type: Literal['noul'] = 'noul', instructions: str | dict[str, JsonValue] | list[JsonValue], criteria: langchain_typesafe.types.NoulCriteria | None = None) -> None
    Ask a binary question and receive the probability that its answer is yes.
    
    Use `Noul` when the probability itself is useful to application code, suc

In [4]:
"""Mechanism smoke test: Coconut-style latent feedback on transformers 5.0.0 / T4 fp16.

The whole project rests on being able to (a) run a forward pass from inputs_embeds,
(b) take the last hidden state, (c) feed it back in as the NEXT input embedding
without ever sampling a token, (d) do that k times, (e) still get gradients through
the entire unrolled loop. If any of those break, the experiment is dead. Test now.
"""
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda().eval()
emb_layer = model.get_input_embeddings()
D = model.config.hidden_size
print(f"loaded {MODEL}  hidden={D}  dtype={next(model.parameters()).dtype}")

ids = tok("Build me a thing to track my gym sessions with friends.", return_tensors="pt").input_ids.cuda()
print("prompt tokens:", ids.shape)

# --- (a) forward from inputs_embeds, with cache ---
x = emb_layer(ids)
out = model(inputs_embeds=x, output_hidden_states=True, use_cache=True)
h = out.hidden_states[-1]
print(f"(a) inputs_embeds OK   hidden_states[-1] {tuple(h.shape)}  n_layers+1={len(out.hidden_states)}")
print(f"    cache type: {type(out.past_key_values).__name__}")

# --- (b..d) k latent steps, no token ever sampled ---
K = 8
cache = out.past_key_values
z = h[:, -1:, :]                      # last hidden state = first "latent thought"
traj = []
for k in range(K):
    o = model(inputs_embeds=z, past_key_values=cache, output_hidden_states=True, use_cache=True)
    cache = o.past_key_values
    z_new = o.hidden_states[-1][:, -1:, :]
    traj.append(z_new.float().squeeze())
    z = z_new
print(f"(b-d) {K} latent steps OK, cache len now {cache.get_seq_length()}")

# does the latent state actually MOVE, or collapse to a fixed point?
import torch.nn.functional as F
T = torch.stack(traj)
step_cos = [F.cosine_similarity(T[i], T[i+1], dim=0).item() for i in range(len(T)-1)]
print("    cos(z_k, z_k+1) per step:", " ".join(f"{c:+.3f}" for c in step_cos))
print(f"    ||z_k|| per step: " + " ".join(f"{t.norm():.1f}" for t in T))

# --- (e) gradients through the unrolled loop ---
model.train()
x2 = emb_layer(ids).detach().requires_grad_(True)
o2 = model(inputs_embeds=x2, output_hidden_states=True, use_cache=False)
z2 = o2.hidden_states[-1][:, -1:, :]
for _ in range(3):
    o2 = model(inputs_embeds=torch.cat([x2, z2], dim=1), output_hidden_states=True, use_cache=False)
    z2 = o2.hidden_states[-1][:, -1:, :]
loss = z2.float().pow(2).mean()
loss.backward()
print(f"(e) grad through 3 unrolled latent steps OK  |grad_input_emb|={x2.grad.norm().item():.4e}")
print(f"    peak GPU mem {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

loaded Qwen/Qwen2.5-0.5B  hidden=896  dtype=torch.float16
prompt tokens: torch.Size([1, 12])
(a) inputs_embeds OK   hidden_states[-1] (1, 12, 896)  n_layers+1=25
    cache type: DynamicCache
(b-d) 8 latent steps OK, cache len now 20
    cos(z_k, z_k+1) per step: +0.963 +0.966 +0.979 +0.985 +0.987 +0.988 +0.987
    ||z_k|| per step: 311.0 305.9 310.2 319.3 326.9 330.9 332.6 332.8
(e) grad through 3 unrolled latent steps OK  |grad_input_emb|=2.9250e+02
    peak GPU mem 1.88 GB


In [6]:
"""Synthetic intent->goals task with ENTAILMENT DEPTH as the difficulty knob.

A user states a few goals explicitly. Others follow by entailment and are never
mentioned: "share with friends" entails "user accounts" entails "persist data".
The model must output the CLOSURE. Depth-3 chains need more inference hops than
depth-1, so if latent compute does anything, depth should be where it shows up.
Ground truth is exact by construction -- no LLM labelling, no distillation.

Seeds are STRATIFIED by depth so the difficulty dial actually has range.
"""
import random, json
from collections import Counter
random.seed(0)

RULES = {
    # depth-3 heads
    "leaderboard": "share_with_friends", "share_with_friends": "user_accounts",
    "sell_items": "payments", "subscriptions": "payments", "refunds": "payments",
    "payments": "user_accounts", "profile_pics": "upload_photos",
    "upload_photos": "file_storage", "reminders": "email_alerts",
    "email_alerts": "user_accounts", "tags": "filters", "filters": "search",
    # depth-2 heads
    "comments": "user_accounts", "weekly_report": "export_csv", "bookmarks": "search",
    # depth-1 heads (and the terminals they hit)
    "user_accounts": "persist_data", "file_storage": "persist_data",
    "export_csv": "persist_data", "search": "persist_data",
    "dark_mode": "settings", "offline_mode": "local_cache", "backup": "persist_data",
}
PHRASE = {
    "leaderboard": "see a leaderboard", "sell_items": "sell stuff",
    "subscriptions": "do monthly subscriptions", "refunds": "handle refunds",
    "profile_pics": "set a profile picture", "reminders": "set reminders",
    "tags": "tag things", "comments": "leave comments",
    "weekly_report": "get a weekly report", "bookmarks": "bookmark things",
    "dark_mode": "use dark mode", "offline_mode": "use it offline",
    "backup": "back everything up", "share_with_friends": "share it with friends",
    "upload_photos": "upload photos", "export_csv": "export to csv",
    "search": "search through it", "filters": "filter results",
    "email_alerts": "get email alerts", "payments": "take payments",
}
DOMAIN = ["gym sessions", "my etsy shop", "book club", "recipes", "freelance invoices",
          "plant watering", "board game nights", "job applications", "study notes"]

def closure(seeds):
    out, frontier = set(seeds), list(seeds)
    while frontier:
        g = frontier.pop()
        if g in RULES and RULES[g] not in out:
            out.add(RULES[g]); frontier.append(RULES[g])
    return out

def depth_of(g):
    d = 0
    while g in RULES: g, d = RULES[g], d + 1
    return d

# stratify: pool seeds by their chain depth, only use ones we can phrase
POOL = {}
for g in RULES:
    if g in PHRASE: POOL.setdefault(depth_of(g), []).append(g)
print("seed pool by depth:", {d: len(v) for d, v in sorted(POOL.items())})

def make(n):
    rows, depths = [], sorted(POOL)
    for i in range(n):
        d = depths[i % len(depths)]                  # balanced strata
        k = min(random.randint(1, 2), len(POOL[d]))
        seeds = random.sample(POOL[d], k)
        full = closure(seeds)
        req = (f"build me something for {random.choice(DOMAIN)} where i can "
               + " and ".join(PHRASE[s] for s in seeds))
        rows.append({"request": req, "goals": sorted(full), "stated": sorted(seeds),
                     "implied": sorted(full - set(seeds)), "depth": d})
    random.shuffle(rows)
    return rows

train, val = make(1600), make(400)
print(json.dumps(train[0], indent=1))
print(f"\ntrain {len(train)} val {len(val)}  |  goal vocab {len(set(RULES)|set(RULES.values()))}")
print("val depth dist:", dict(sorted(Counter(r['depth'] for r in val).items())))
for d in sorted(POOL):
    sub = [r for r in val if r['depth'] == d]
    print(f"  depth {d}: mean implied goals (needs inference) = "
          f"{sum(len(r['implied']) for r in sub)/max(len(sub),1):.2f}")


seed pool by depth: {1: 5, 2: 8, 3: 7}
{
 "request": "build me something for freelance invoices where i can export to csv and use it offline",
 "goals": [
  "export_csv",
  "local_cache",
  "offline_mode",
  "persist_data"
 ],
 "stated": [
  "export_csv",
  "offline_mode"
 ],
 "implied": [
  "local_cache",
  "persist_data"
 ],
 "depth": 1
}

train 1600 val 400  |  goal vocab 25
val depth dist: {1: 134, 2: 133, 3: 133}
  depth 1: mean implied goals (needs inference) = 1.30
  depth 2: mean implied goals (needs inference) = 2.47
  depth 3: mean implied goals (needs inference) = 3.68


In [7]:
import torch, time, gc, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

for v in ["model","emb_layer","out","o","o2","cache"]: globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

MODEL, K_TRAIN, BS, STEPS, LR, TIME_CAP = "Qwen/Qwen2.5-0.5B", 4, 16, 300, 2e-4, 900
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda()
base.config.use_cache = False
model = get_peft_model(base, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
for n, p in model.named_parameters():
    if p.requires_grad: p.data = p.data.float()
emb = model.get_input_embeddings()
print("trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad))

PROMPT = "user request: {r}\ngoals:"
def encode(rows):
    tok.padding_side = "left"
    p = tok([PROMPT.format(r=x["request"]) for x in rows], return_tensors="pt", padding=True)
    tok.padding_side = "right"
    t = tok([" " + ", ".join(x["goals"]) + tok.eos_token for x in rows],
            return_tensors="pt", padding=True, add_special_tokens=False)
    return (p.input_ids.cuda(), p.attention_mask.cuda(),
            t.input_ids.cuda(), t.attention_mask.cuda())

def latent_run(pid, pmask, k, tid=None, tmask=None):
    """prompt -> k latent steps -> (optional) target. Returns logits over target span."""
    seq, mask = emb(pid), pmask
    for _ in range(k):
        h = model(inputs_embeds=seq, attention_mask=mask,
                  output_hidden_states=True, use_cache=False).hidden_states[-1]
        seq = torch.cat([seq, h[:, -1:, :]], 1)
        mask = torch.cat([mask, torch.ones_like(mask[:, :1])], 1)
    if tid is None: return seq, mask
    full = torch.cat([seq, emb(tid)], 1)
    fmask = torch.cat([mask, tmask], 1)
    logits = model(inputs_embeds=full, attention_mask=fmask, use_cache=False).logits
    lab = torch.full(fmask.shape, -100, device=fmask.device, dtype=torch.long)
    lab[:, seq.shape[1]:] = tid.masked_fill(tmask == 0, -100)
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)).float(),
                           lab[:, 1:].reshape(-1), ignore_index=-100)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scaler = torch.amp.GradScaler("cuda")
sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=STEPS, pct_start=0.1)
model.train(); t0 = time.time(); hist = []
for step in range(STEPS):
    rows = [train[(step * BS + i) % len(train)] for i in range(BS)]
    with torch.amp.autocast("cuda", dtype=torch.float16):
        loss = latent_run(*encode(rows), k=K_TRAIN) if False else None
    pid, pmask, tid, tmask = encode(rows)
    with torch.amp.autocast("cuda", dtype=torch.float16):
        loss = latent_run(pid, pmask, K_TRAIN, tid, tmask)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0)
    scaler.step(opt); scaler.update(); sched.step()
    hist.append(loss.item())
    if step % 50 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  loss {sum(hist[-50:])/len(hist[-50:]):.4f}  "
              f"{time.time()-t0:.0f}s  {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    if time.time() - t0 > TIME_CAP:
        print(f"TIME CAP hit at step {step}"); break
print(f"done in {time.time()-t0:.0f}s")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable: 8798208


/tmp/ipykernel_102/159700856.py:64: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


step    0  loss 7.0382  1s  8.4GB
step   50  loss 1.7846  55s  8.9GB
step  100  loss 0.0320  109s  9.0GB
step  150  loss 0.0058  162s  9.0GB
step  200  loss 0.0004  214s  9.0GB
step  250  loss 0.0003  269s  9.0GB
step  299  loss 0.0002  321s  9.0GB
done in 321s


In [13]:
"""Two measurements:
  (1) COLLAPSE DIAGNOSTIC on the trained model -- untrained baseline was
      cos 0.963 -> 0.988 (fixed point by step ~5). Did training un-collapse it?
  (2) k-SWEEP -- decode goals at k in {0,1,2,4,8}, scored by depth stratum.
      k=0 is the matched control: same weights, same params, no thinking.
Primary metric is IMPLIED-goal recall: the goals never stated, which require
entailment hops. Stated-goal recall is the sanity control and should stay flat.
"""
import torch, time, torch.nn.functional as F
from collections import defaultdict
model.eval()

# ---------- (1) collapse diagnostic ----------
with torch.no_grad():
    pid, pmask, _, _ = encode(val[:8])
    seq, mask = emb(pid), pmask
    zs = []
    for _ in range(8):
        h = model(inputs_embeds=seq, attention_mask=mask,
                  output_hidden_states=True, use_cache=False).hidden_states[-1]
        z = h[:, -1:, :]; zs.append(z.float())
        seq = torch.cat([seq, z], 1)
        mask = torch.cat([mask, torch.ones_like(mask[:, :1])], 1)
cos = [F.cosine_similarity(zs[i].flatten(1), zs[i+1].flatten(1), dim=1).mean().item()
       for i in range(len(zs)-1)]
print("TRAINED   cos(z_k,z_k+1):", " ".join(f"{c:+.3f}" for c in cos))
print("untrained cos(z_k,z_k+1): +0.963 +0.966 +0.979 +0.985 +0.987 +0.988 +0.987")
print("norms:", " ".join(f"{z.norm(dim=-1).mean():.0f}" for z in zs))

# ---------- (2) k-sweep ----------
@torch.no_grad()
def decode(rows, k, max_new=48):
    pid, pmask, _, _ = encode(rows)
    seq, mask = emb(pid), pmask
    for _ in range(k):
        h = model(inputs_embeds=seq, attention_mask=mask,
                  output_hidden_states=True, use_cache=False).hidden_states[-1]
        seq = torch.cat([seq, h[:, -1:, :]], 1)
        mask = torch.cat([mask, torch.ones_like(mask[:, :1])], 1)
    outs = torch.zeros(len(rows), 0, dtype=torch.long, device=seq.device)
    done = torch.zeros(len(rows), dtype=torch.bool, device=seq.device)
    for _ in range(max_new):
        lg = model(inputs_embeds=seq, attention_mask=mask, use_cache=False).logits[:, -1]
        nxt = lg.argmax(-1)
        nxt = torch.where(done, torch.full_like(nxt, tok.pad_token_id), nxt)
        outs = torch.cat([outs, nxt[:, None]], 1)
        done |= nxt.eq(tok.eos_token_id)
        if done.all(): break
        seq = torch.cat([seq, emb(nxt[:, None])], 1)
        mask = torch.cat([mask, torch.ones_like(mask[:, :1])], 1)
    return [set(g.strip() for g in s.split(",") if g.strip())
            for s in tok.batch_decode(outs, skip_special_tokens=True)]

sub = [r for d in (1, 2, 3) for r in [x for x in val if x["depth"] == d][:50]]
print(f"\neval n={len(sub)}  (50 per depth)")
res = {}
t0 = time.time()
for k in (0, 1, 2, 4, 8):
    agg = defaultdict(lambda: [0, 0, 0, 0])   # depth -> [impl_hit, impl_tot, exact, n]
    for i in range(0, len(sub), 25):
        batch = sub[i:i+25]
        for r, pred in zip(batch, decode(batch, k)):
            a = agg[r["depth"]]
            imp = set(r["implied"])
            a[0] += len(imp & pred); a[1] += len(imp)
            a[2] += int(pred == set(r["goals"])); a[3] += 1
    line = []
    for d in (1, 2, 3):
        h, t, e, n = agg[d]
        line.append(f"d{d} impl={h/max(t,1):.3f} exact={e/n:.2f}")
    allh = sum(agg[d][0] for d in agg); allt = sum(agg[d][1] for d in agg)
    res[k] = allh / max(allt, 1)
    print(f"k={k}  IMPLIED-recall(all)={res[k]:.3f}  | " + "  ".join(line)
          + f"  [{time.time()-t0:.0f}s]")
print("\nk-curve (implied recall):", {k: round(v, 3) for k, v in res.items()})


TRAINED   cos(z_k,z_k+1): +0.954 +0.984 +0.993 +0.996 +0.998 +0.998 +0.999
untrained cos(z_k,z_k+1): +0.963 +0.966 +0.979 +0.985 +0.987 +0.988 +0.987
norms: 226 236 242 246 250 253 255 257

eval n=150  (50 per depth)
k=0  IMPLIED-recall(all)=0.964  | d1 impl=0.986 exact=0.96  d2 impl=0.982 exact=0.92  d3 impl=0.945 exact=0.76  [22s]
k=1  IMPLIED-recall(all)=0.968  | d1 impl=0.986 exact=0.96  d2 impl=0.982 exact=0.92  d3 impl=0.954 exact=0.78  [47s]
k=2  IMPLIED-recall(all)=0.970  | d1 impl=0.986 exact=0.96  d2 impl=0.988 exact=0.94  d3 impl=0.954 exact=0.76  [72s]
k=4  IMPLIED-recall(all)=0.970  | d1 impl=0.986 exact=0.96  d2 impl=0.982 exact=0.90  d3 impl=0.958 exact=0.80  [102s]
k=8  IMPLIED-recall(all)=0.968  | d1 impl=0.986 exact=0.96  d2 impl=0.982 exact=0.92  d3 impl=0.954 exact=0.78  [139s]

k-curve (implied recall): {0: 0.964, 1: 0.968, 2: 0.97, 4: 0.97, 8: 0.968}


In [9]:
"""RUN 2 -- the control that decides run 1.

Run 1 trained only at k=4, so evaluating at k=0/1 was off-distribution and the
depth-3 gradient (0.82 -> 1.00) could have been train/test k-matching rather than
compute. Here k is sampled uniformly per step from the same set we evaluate on,
so no test-time k is off-distribution. Everything else is held fixed.

If the depth-3 gradient survives -> the effect is real.
If it vanishes -> run 1 was distribution shift, and that is the finding.
"""
import torch, time, gc, random, torch.nn.functional as F
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

del model, base
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

K_SET, BS, STEPS, LR, TIME_CAP = [0, 1, 2, 4, 8], 8, 600, 2e-4, 1100
random.seed(1); torch.manual_seed(1)

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda()
base.config.use_cache = False
model = get_peft_model(base, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
for _, p in model.named_parameters():
    if p.requires_grad: p.data = p.data.float()
emb = model.get_input_embeddings()

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scaler = torch.amp.GradScaler("cuda")
sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=STEPS, pct_start=0.1)
model.train(); t0 = time.time(); hist = []; per_k = {k: [] for k in K_SET}

for step in range(STEPS):
    k = K_SET[step % len(K_SET)]                      # balanced over k, not random
    rows = [train[(step * BS + i) % len(train)] for i in range(BS)]
    pid, pmask, tid, tmask = encode(rows)
    with torch.amp.autocast("cuda", dtype=torch.float16):
        loss = latent_run(pid, pmask, k, tid, tmask)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    scaler.step(opt); scaler.update(); sched.step()
    hist.append(loss.item()); per_k[k].append(loss.item())
    if step % 100 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  loss {sum(hist[-100:])/len(hist[-100:]):.4f}  "
              f"{time.time()-t0:.0f}s  {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    if time.time() - t0 > TIME_CAP:
        print(f"TIME CAP at step {step}"); break

print(f"done in {time.time()-t0:.0f}s")
print("final train loss by k:", {k: round(sum(v[-20:])/max(len(v[-20:]),1), 4)
                                 for k, v in per_k.items() if v})


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

step    0  loss 4.6865  0s  4.1GB
step  100  loss 1.3932  87s  7.6GB
step  200  loss 0.0316  175s  7.7GB
step  300  loss 0.0039  261s  7.7GB
step  400  loss 0.0013  346s  7.7GB
step  500  loss 0.0002  431s  7.7GB
step  599  loss 0.0002  514s  7.7GB
done in 514s
final train loss by k: {0: 0.0002, 1: 0.0001, 2: 0.0001, 4: 0.0001, 8: 0.0002}


In [11]:
"""HARD TASK v2 -- rules are RANDOMISED PER EXAMPLE and given in the prompt.

In v1 the entailment graph was global and fixed, so 8.8M LoRA params simply
memorised it and k=0 hit 1.00. Here every example carries its own random rule set,
so the graph cannot be stored in the weights: the model must read the rules and
traverse them at inference time. Memorisation is ruled out by construction, and
the eval rule sets are novel by construction too.

Depth d = number of hops from the stated goal to the end of its chain.
Distractor rules are present but never reachable from the stated goals.
"""
import random, json
from collections import Counter
rng = random.Random(7)

NAMES = ["accounts","payments","storage","search","alerts","export","tags","comments",
         "photos","profiles","invites","roles","audit","backup","sync","cache",
         "webhooks","ratings","calendar","maps","chat","polls","badges","streaks",
         "reminders","imports","themes","locales","quotas","billing","refunds","tickets",
         "reports","charts","filters","archive"]
DOMAIN = ["gym sessions","my etsy shop","book club","recipes","freelance invoices",
          "plant watering","board game nights","job applications","study notes"]
CHAIN_LEN, N_CHAINS = 4, 3

def make_example(d):
    names = rng.sample(NAMES, CHAIN_LEN * (N_CHAINS + 1))
    chains = [names[i*CHAIN_LEN:(i+1)*CHAIN_LEN] for i in range(N_CHAINS + 1)]
    rules = {}
    for ch in chains:
        for a, b in zip(ch, ch[1:]): rules[a] = b
    live = chains[:N_CHAINS]                      # chains[-1] is pure distractor
    n_seed = rng.randint(1, 2)
    picked = rng.sample(live, n_seed)
    seeds = [ch[CHAIN_LEN - 1 - d] for ch in picked]   # exactly d hops remain
    full = set(seeds)
    for s in seeds:
        g = s
        while g in rules:
            g = rules[g]; full.add(g)
    rule_txt = ", ".join(f"{a}>{b}" for a, b in
                         sorted(rules.items(), key=lambda kv: rng.random()))
    req = (f"rules: {rule_txt} | for {rng.choice(DOMAIN)} the user wants: "
           + ", ".join(seeds))
    return {"request": req, "goals": sorted(full), "stated": sorted(seeds),
            "implied": sorted(full - set(seeds)), "depth": d}

def make(n):
    rows = [make_example([1, 2, 3][i % 3]) for i in range(n)]
    rng.shuffle(rows); return rows

train, val = make(4800), make(400)
e = train[0]
print(json.dumps({k: e[k] for k in ("request","goals","implied","depth")}, indent=1)[:600])
print("\nval depth dist:", dict(sorted(Counter(r['depth'] for r in val).items())))
for d in (1, 2, 3):
    sub = [r for r in val if r['depth'] == d]
    print(f"  depth {d}: mean implied = {sum(len(r['implied']) for r in sub)/len(sub):.2f}")

# how long is the prompt now? drives memory at k=8
tok.padding_side = "left"
L = tok([PROMPT.format(r=r["request"]) for r in val[:64]], padding=True).input_ids
print(f"\nprompt tokens: {len(L[0])} (was ~30 in v1)")
# rule sets in val that also appear in train -> must be ~0
tr = {r["request"].split(" | ")[0] for r in train}
print("val rule-sets seen in train:", sum(r["request"].split(" | ")[0] in tr for r in val), "/", len(val))


{
 "request": "rules: payments>archive, export>polls, archive>tickets, sync>storage, ratings>reminders, storage>audit, reminders>export, webhooks>sync, roles>billing, billing>tags, tickets>streaks, tags>badges | for freelance invoices the user wants: sync, billing",
 "goals": [
  "audit",
  "badges",
  "billing",
  "storage",
  "sync",
  "tags"
 ],
 "implied": [
  "audit",
  "badges",
  "storage",
  "tags"
 ],
 "depth": 2
}

val depth dist: {1: 134, 2: 133, 3: 133}
  depth 1: mean implied = 1.46
  depth 2: mean implied = 2.92
  depth 3: mean implied = 4.56

prompt tokens: 73 (was ~30 in v1)
val rule-sets seen in train: 0 / 400


In [12]:
"""RUN 3 -- hard task (per-example random rules) + the run-2 control from the start.

k is balanced over {0,1,2,4,8} during training, so every test-time k is
in-distribution. This is the run that can actually answer the question: with
memorisation ruled out and no k-mismatch confound, does latent compute buy
anything on the deep-chain items?
"""
import torch, time, gc, random
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

del model, base
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

K_SET, BS, STEPS, LR, TIME_CAP = [0, 1, 2, 4, 8], 6, 750, 2e-4, 1050
random.seed(2); torch.manual_seed(2)

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda()
base.config.use_cache = False
model = get_peft_model(base, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
for _, p in model.named_parameters():
    if p.requires_grad: p.data = p.data.float()
emb = model.get_input_embeddings()

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scaler = torch.amp.GradScaler("cuda")
sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=STEPS, pct_start=0.1)
model.train(); t0 = time.time(); hist = []; per_k = {k: [] for k in K_SET}

for step in range(STEPS):
    k = K_SET[step % len(K_SET)]
    rows = [train[(step * BS + i) % len(train)] for i in range(BS)]
    pid, pmask, tid, tmask = encode(rows)
    with torch.amp.autocast("cuda", dtype=torch.float16):
        loss = latent_run(pid, pmask, k, tid, tmask)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    scaler.step(opt); scaler.update(); sched.step()
    hist.append(loss.item()); per_k[k].append(loss.item())
    if step % 125 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  loss {sum(hist[-125:])/len(hist[-125:]):.4f}  "
              f"{time.time()-t0:.0f}s  {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    if time.time() - t0 > TIME_CAP:
        print(f"TIME CAP at step {step}"); break

print(f"done in {time.time()-t0:.0f}s   steps run: {len(hist)}")
print("final train loss by k:", {k: round(sum(v[-15:])/max(len(v[-15:]),1), 4)
                                 for k, v in per_k.items() if v})


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/tmp/ipykernel_102/941711800.py:42: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


step    0  loss 2.9999  0s  4.7GB
step  125  loss 1.4360  111s  12.1GB
step  250  loss 0.5069  218s  12.1GB
step  375  loss 0.2895  325s  12.2GB
step  500  loss 0.1647  432s  12.2GB
step  625  loss 0.0937  538s  12.2GB
step  749  loss 0.0682  646s  12.2GB
done in 646s   steps run: 750
final train loss by k: {0: 0.0482, 1: 0.0442, 2: 0.0752, 4: 0.0753, 8: 0.08}
